# IDM-VTON Test — Lookzi benchmark payrlar uchun

**Maqsad:** IDM-VTON ni CatVTON bilan taqqoslash (lower, overall, upper).

**Tartib:** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 ketma-ket run qiling.

> Alohida Colab session'da oching — CatVTON Colab'dan mustaqil.

In [ ]:
# Cell 1 — GPU tekshirish
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOQ'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2 — Drive mount
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Cell 3 — IDM-VTON va Lookzi clone
%cd /content
!rm -rf /content/IDM-VTON
!git clone https://github.com/yisol/IDM-VTON.git /content/IDM-VTON

!rm -rf /content/Lookzi
!git clone https://github.com/Mohamed-Kudratov/Lookzi.git /content/Lookzi

%cd /content/IDM-VTON
print("Clone tayyor.")

In [ ]:
# Cell 4 — IDM-VTON dependencies
!pip install -q omegaconf einops
!pip install -q transformers==4.46.3 accelerate diffusers==0.25.0
!pip install -q torchvision opencv-python Pillow huggingface_hub
!pip install -q detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu121/torch2.5/index.html

import torch
print(f"torch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print("Dependencies tayyor.")

In [ ]:
# Cell 5 — IDM-VTON modelini Drive'ga yuklab olish
# Birinchi marta ~8-10 GB, ~20-30 daqiqa. Keyingi safar 1-2 daqiqa.
import os, shutil
from huggingface_hub import snapshot_download

DRIVE_IDM = "/content/drive/MyDrive/Lookzi/hf_models/idm-vton"
LOCAL_CKPT = "/content/IDM-VTON/ckpt/IDM-VTON"

os.makedirs(os.path.dirname(LOCAL_CKPT), exist_ok=True)

if not os.path.exists(DRIVE_IDM):
    print("IDM-VTON yuklanmoqda (~8GB) — ~20-30 daqiqa kutiladi...")
    snapshot_download(repo_id="yisol/IDM-VTON", local_dir=DRIVE_IDM)
    print("Yuklandi va Drive'ga saqlandi.")
else:
    print("Drive'da mavjud — yuklanmaydi.")

# Symlink
if os.path.islink(LOCAL_CKPT): os.unlink(LOCAL_CKPT)
if os.path.exists(LOCAL_CKPT) and not os.path.islink(LOCAL_CKPT):
    shutil.rmtree(LOCAL_CKPT)
os.symlink(DRIVE_IDM, LOCAL_CKPT)
print(f"Symlink: {LOCAL_CKPT} -> {DRIVE_IDM}")

# Model fayllari borligini tekshirish
unet_path = os.path.join(DRIVE_IDM, "unet")
if os.path.exists(unet_path):
    print("UNet topildi.")
else:
    print("XATO: UNet topilmadi. Yuklash to'liq bo'lmagan bo'lishi mumkin.")

In [ ]:
# Cell 6 — IDM-VTON pipeline yuklash va bitta test
import sys, os, torch
from PIL import Image

sys.path.insert(0, '/content/IDM-VTON')

print("IDM-VTON pipeline yuklanmoqda (1-2 daqiqa)...")

try:
    # IDM-VTON Gradio demo pipeline
    from gradio_demo.app import start_tryon
    IDM_MODE = "gradio"
    print("Pipeline yuklandi (gradio_demo rejimi).")
except Exception as e:
    print(f"gradio_demo ishlamadi: {e}")
    IDM_MODE = "failed"

if IDM_MODE == "failed":
    print("\nTavsiya: Quyida xato xabarini ko'ring va muammoni aniqlang.")
else:
    # Bitta test rasmda sinab ko'rish
    print("\nBitta test rasmda sinab ko'rilmoqda...")
    test_person  = "/content/Lookzi/resource/demo/example/person/men/Yifeng_0.png"
    test_garment = "/content/Lookzi/resource/demo/example/condition/lower/men_pants.png"

    person_img  = Image.open(test_person).convert('RGB')
    garment_img = Image.open(test_garment).convert('RGB')

    try:
        result, mask = start_tryon(
            {"background": person_img, "layers": [], "composite": None},
            garment_img,
            garment_des="",
            is_checked=True,        # auto mask
            is_checked_crop=False,
            denoise_steps=20,
            seed=42,
        )
        print("Test muvaffaqiyatli! Natija ko'rsatilmoqda...")
        from IPython.display import display
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 3, figsize=(12, 6))
        for ax, img, t in zip(axes, [person_img, garment_img, result], ["Person", "Garment", "IDM-VTON natija"]):
            ax.imshow(img.resize((256, 384))); ax.set_title(t, fontsize=13); ax.axis('off')
        plt.tight_layout(); plt.show()
        IDM_READY = True
    except Exception as e:
        print(f"Test xatosi: {e}")
        IDM_READY = False

print(f"\nIDM tayyor: {IDM_READY if IDM_MODE != 'failed' else False}")

In [ ]:
# Cell 7 — Barcha benchmark pairlar uchun IDM-VTON test
# Cell 6 muvaffaqiyatli bo'lsa run qiling.
import json, os
from PIL import Image
import matplotlib.pyplot as plt
import torch

LOOKZI_ROOT  = "/content/Lookzi"
OUTPUT_DIR   = "/content/drive/MyDrive/Lookzi/idm_test_results"
CATVTON_DIR  = "/content/drive/MyDrive/Lookzi/eval_logs/outputs"  # CatVTON natijalari
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Qaysi pairlarni test qilish kerak?
# None = hammasi (C01-C08)
# List = faqat belgilanganlari
TEST_IDS = None  # yoki: ["C04", "C05", "C07", "C08"]

CATEGORY_MAP = {
    "upper":   "upper_body",
    "lower":   "lower_body",
    "overall": "dresses",
}

with open(f"{LOOKZI_ROOT}/benchmark/catalog_pairs.json") as f:
    pairs = json.load(f)["pairs"]

if TEST_IDS:
    pairs = [p for p in pairs if p["id"] in TEST_IDS]

print(f"{len(pairs)} ta test qilinadi.\n")

for pair in pairs:
    pid          = pair["id"]
    tag          = pair["tag"]
    person_path  = f"{LOOKZI_ROOT}/{pair['person']}"
    garment_path = f"{LOOKZI_ROOT}/{pair['garment']}"
    cloth_type   = pair["cloth_type"]
    save_path    = f"{OUTPUT_DIR}/{pid}_idm.png"

    print(f"{'='*55}")
    print(f"{pid} | {cloth_type} | {tag}")

    # Oldin saqlangan bo'lsa skip
    if os.path.exists(save_path):
        print(f"  Oldin saqlangan — skip.")
        idm_result = Image.open(save_path)
    else:
        print(f"  IDM-VTON ishlamoqda...")
        torch.cuda.empty_cache()
        try:
            person_img  = Image.open(person_path).convert('RGB')
            garment_img = Image.open(garment_path).convert('RGB')
            idm_result, _ = start_tryon(
                {"background": person_img, "layers": [], "composite": None},
                garment_img,
                garment_des="",
                is_checked=True,
                is_checked_crop=False,
                denoise_steps=20,
                seed=42,
            )
            idm_result.save(save_path)
            print(f"  Saqlandi: {save_path}")
        except Exception as e:
            print(f"  XATO: {e}")
            idm_result = None

    # --- Ko'rsatish ---
    person_img  = Image.open(person_path).convert('RGB').resize((256, 384))
    garment_img = Image.open(garment_path).convert('RGB').resize((256, 384))

    # CatVTON natijasi bormi?
    catvton_path = f"{CATVTON_DIR}/{pid}_result.png"
    has_catvton  = os.path.exists(catvton_path)

    n_cols = 2 + (1 if idm_result else 0) + (1 if has_catvton else 0)
    fig, axes = plt.subplots(1, n_cols, figsize=(n_cols * 3.5, 6))
    if n_cols == 1: axes = [axes]
    fig.suptitle(f"{pid} — {cloth_type} — {tag}", fontsize=11, fontweight='bold')

    col = 0
    axes[col].imshow(person_img);  axes[col].set_title("Person");  axes[col].axis('off'); col += 1
    axes[col].imshow(garment_img); axes[col].set_title("Garment"); axes[col].axis('off'); col += 1

    if has_catvton:
        cat = Image.open(catvton_path).convert('RGB').resize((256, 384))
        axes[col].imshow(cat); axes[col].set_title("CatVTON"); axes[col].axis('off'); col += 1

    if idm_result:
        axes[col].imshow(idm_result.resize((256, 384)))
        axes[col].set_title("IDM-VTON"); axes[col].axis('off')

    plt.tight_layout()
    plt.show()

print("\nBarcha testlar tugadi.")
print(f"Natijalar: {OUTPUT_DIR}")